# SGDRegressor — Boston Housing (MEDV)

Objectif : prédire **MEDV** (prix médian) à partir des caractéristiques des quartiers.

Ce notebook est structuré en 2 parties :
1) **Mise en place** (données, visualisation, normalisation, entraînement de base)
2) **Travail étudiant** : modifier les hyperparamètres et analyser l'impact

In [ ]:
# Installation (utile hors Colab)
# Sur Colab, ces libs sont déjà disponibles
# Si besoin, décommenter :
# !pip -q install -r requirements.txt

## 1) Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import SGDRegressor

pd.options.display.max_columns = None

## 2) Chargement des données

Le fichier `public/data/original_boston.csv` contient les features et la cible `MEDV`.

In [ ]:
DATA_PATH = "../data/original_boston.csv"  # relatif au dossier notebooks


def load_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    return df


df = load_data(DATA_PATH)
df.head()

## 3) Visualisation rapide (histogrammes)

On observe la distribution de quelques variables clés.

In [ ]:
FEATURES_TO_PLOT = ["RM", "LSTAT", "PTRATIO", "TAX", "NOX"]


def plot_histograms(df: pd.DataFrame, features):
    df[features].hist(figsize=(10, 6), bins=20)
    plt.suptitle("Histogrammes des features", y=1.02)
    plt.show()


plot_histograms(df, FEATURES_TO_PLOT)

## 4) Préparation des données

- Séparer X et y
- Normaliser (StandardScaler)
- Split train / test

In [ ]:
TARGET = "MEDV"


def split_features_target(df: pd.DataFrame, target: str):
    X = df.drop(columns=[target]).values
    y = df[target].values
    return X, y


def normalize_split(X, y, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    return X_train, X_test, y_train, y_test


X, y = split_features_target(df, TARGET)
X_train, X_test, y_train, y_test = normalize_split(X, y)

## 5) Entraînement mini-batch (SGD)

On entraîne avec `partial_fit` pour pouvoir choisir le **batch size** et suivre l'évolution
sur **train** et **test** à chaque époque.

In [ ]:
CONFIG = {
    "epochs": 40,
    "batch_size": 16,
    "eta0": 0.01,
    "alpha": 0.0001,
    "random_state": 42,
}


def train_sgd_minibatch(X_train, y_train, X_val, y_val, config):
    model = SGDRegressor(
        loss="squared_error",
        penalty="l2",
        alpha=config["alpha"],
        learning_rate="constant",
        eta0=config["eta0"],
        random_state=config["random_state"],
    )

    rng = np.random.default_rng(config["random_state"])
    n = X_train.shape[0]
    history = {"train_rmse": [], "test_rmse": []}

    for epoch in range(config["epochs"]):
        indices = rng.permutation(n)
        for start in range(0, n, config["batch_size"]):
            batch_idx = indices[start : start + config["batch_size"]]
            X_batch = X_train[batch_idx]
            y_batch = y_train[batch_idx]
            model.partial_fit(X_batch, y_batch)

        # RMSE train/test
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_val)
        train_rmse = mean_squared_error(y_train, train_pred, squared=False)
        test_rmse = mean_squared_error(y_val, test_pred, squared=False)

        history["train_rmse"].append(train_rmse)
        history["test_rmse"].append(test_rmse)

    return model, history

## 6) Visualisation des RMSE

In [ ]:
def plot_rmse_history(history, title="RMSE par époque"):
    plt.figure(figsize=(7, 4))
    plt.plot(history["train_rmse"], label="Train")
    plt.plot(history["test_rmse"], label="Test")
    plt.title(title)
    plt.xlabel("Époque")
    plt.ylabel("RMSE")
    plt.legend()
    plt.grid(True)
    plt.show()

## 7) Entraînement de base (batch_size = 16)

In [ ]:
model, history = train_sgd_minibatch(X_train, y_train, X_test, y_test, CONFIG)
plot_rmse_history(history, "RMSE — configuration de base")

print("RMSE train (final):", history["train_rmse"][-1])
print("RMSE test  (final):", history["test_rmse"][-1])

# Partie 2 — Travail étudiant

Objectif : **modifier les hyperparamètres** et observer l'impact sur les courbes RMSE.

Vous devez **écrire du code** dans les cellules ci-dessous en réutilisant les fonctions existantes.

## Étape 1 — Tester différents learning rates

Consigne :
- créer une liste de valeurs pour `eta0` (ex: `[0.1, 0.01, 0.001]`)
- pour chaque valeur, entraîner le modèle et tracer les courbes RMSE

In [ ]:
# TODO: definir des valeurs de learning rate
learning_rates = [0.1, 0.01, 0.001]

for lr in learning_rates:
    cfg = {**CONFIG, "eta0": lr}
    model, history = train_sgd_minibatch(X_train, y_train, X_test, y_test, cfg)
    plot_rmse_history(history, f"RMSE — eta0={lr}")

## Étape 2 — Tester différents batch sizes

Consigne :
- essayer des valeurs comme 1, 8, 16, 64
- comparer les courbes train/test

In [ ]:
# TODO: definir des tailles de batch
batch_sizes = [1, 8, 16, 64]

for bs in batch_sizes:
    cfg = {**CONFIG, "batch_size": bs}
    model, history = train_sgd_minibatch(X_train, y_train, X_test, y_test, cfg)
    plot_rmse_history(history, f"RMSE — batch_size={bs}")

## Étape 3 — Tester la régularisation L2 (alpha)

Consigne :
- tester plusieurs valeurs de `alpha` (ex: `0.0`, `0.0001`, `0.001`, `0.01`)
- observer l'effet sur le sur-apprentissage (train vs test)

In [ ]:
# TODO: definir des valeurs de regularisation
alphas = [0.0, 0.0001, 0.001, 0.01]

for a in alphas:
    cfg = {**CONFIG, "alpha": a}
    model, history = train_sgd_minibatch(X_train, y_train, X_test, y_test, cfg)
    plot_rmse_history(history, f"RMSE — alpha={a}")